In [73]:
import pandas as pd

df = pd.read_csv("quality_data.csv")

# Create input text
df["input_text"] = (
    "narration: " + df["narration"].astype(str) +
    " | mode: " + df["mode"].astype(str) +
    " | type: " + df["type"].astype(str)
)

df["input_text"] = df["input_text"].str.lower()

# Create label
df["label"] = df["category"] + "_" + df["subcategory"]

df = df[["input_text", "label"]]
df.head()

,input_text,label
0,narration: interest from fixed deposit | mode:...,investment_fd_interest
1,narration: fd interest payout | mode: neft | t...,investment_fd_interest
2,narration: fixed deposit interest | mode: neft...,investment_fd_interest
3,narration: bank interest received | mode: neft...,investment_fd_interest
4,narration: fd maturity interest | mode: neft |...,investment_fd_interest


In [74]:
from sklearn.model_selection import train_test_split

# X_train, X_test, y_train, y_test = train_test_split(
#     df["input_text"],
#     df["label"],
#     test_size=0.2,
#     random_state=42,
#     stratify=df["label"]  # important for balance
# )
X_train = df["input_text"]
y_train = df["label"]

X_test = df["input_text"]
y_test = df["label"]

In [75]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

tfidf_model = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", LogisticRegression(max_iter=1000))
])

tfidf_model.fit(X_train, y_train)

y_pred_tfidf = tfidf_model.predict(X_test)

In [76]:
print(y_test.value_counts())

label
expense_loan              12
expense_shopping          12
investment_fd_interest    10
expense_transport         10
investment_fd_booking      8
income_salary              8
investment_stocks          8
expense_food               8
expense_bills              8
expense_insurance          8
expense_health             7
Name: count, dtype: int64


In [77]:
import numpy as np

probs = tfidf_model.predict_proba(X_test)
confidence_scores = np.max(probs, axis=1)

In [78]:
from gliner import GLiNER

gliner_model = GLiNER.from_pretrained("urchade/gliner_base")

labels = list(set(y_train))

c:\Users\ACER\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\utils\_validators.py:189: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

In [79]:
def explain_tfidf_pipeline(text, pipeline, top_n=3):
    import numpy as np
    
    vectorizer = pipeline.named_steps["tfidf"]
    model = pipeline.named_steps["clf"]
    
    vec = vectorizer.transform([text])
    feature_names = np.array(vectorizer.get_feature_names_out())
    scores = vec.toarray()[0]
    
    if scores.sum() == 0:
        return "No strong words → prediction based on weak similarity"
    
    # Top words from input
    top_indices = scores.argsort()[-top_n:][::-1]
    top_words = feature_names[top_indices]
    
    # Get predicted class
    pred_class = pipeline.predict([text])[0]
    
    return f"Words [{', '.join(top_words)}] influenced prediction → classified as '{pred_class}'"

In [80]:
def gliner_predict(text):
    entities = gliner_model.predict_entities(text, labels)

    if entities:
        return entities[0]["label"], entities[0]["score"]
    return "unknown", 0.0


y_pred_gliner = []
gliner_conf = []

for text in X_test:
    label, score = gliner_predict(text)
    y_pred_gliner.append(label)
    gliner_conf.append(score)

In [81]:
from sklearn.metrics import classification_report
import pandas as pd

report = classification_report(y_test, y_pred_tfidf, output_dict=True)

report_df = pd.DataFrame(report).transpose()

# Add F2 score manually
def f2_score(p, r):
    if (p + r) == 0:
        return 0
    return (5 * p * r) / (4 * p + r)

report_df["f2_score"] = report_df.apply(
    lambda row: f2_score(row["precision"], row["recall"]),
    axis=1
)

report_df = report_df.round(3)

print(report_df)

                        precision  recall  f1-score  support  f2_score
expense_bills               0.875   0.875     0.875     8.00     0.875
expense_food                1.000   0.875     0.933     8.00     0.897
expense_health              1.000   1.000     1.000     7.00     1.000
expense_insurance           1.000   1.000     1.000     8.00     1.000
expense_loan                1.000   1.000     1.000    12.00     1.000
expense_shopping            1.000   1.000     1.000    12.00     1.000
expense_transport           0.909   1.000     0.952    10.00     0.980
income_salary               1.000   1.000     1.000     8.00     1.000
investment_fd_booking       1.000   1.000     1.000     8.00     1.000
investment_fd_interest      1.000   1.000     1.000    10.00     1.000
investment_stocks           1.000   1.000     1.000     8.00     1.000
accuracy                    0.980   0.980     0.980     0.98     0.980
macro avg                   0.980   0.977     0.978    99.00     0.978
weight

In [82]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred_tfidf)
print("\nAccuracy:", round(accuracy, 3))


Accuracy: 0.98


In [83]:
print("\nBest Performing Category:")
print(report_df["f1-score"].idxmax(), report_df["f1-score"].max())

print("\nWorst Performing Category:")
print(report_df["f1-score"].idxmin(), report_df["f1-score"].min())


Best Performing Category:
expense_health 1.0

Worst Performing Category:
expense_bills 0.875


In [84]:
result_df = pd.DataFrame({
    "input_text": X_test,
    "actual_label": y_test,
    "predicted_label": y_pred_tfidf,
    "confidence": confidence_scores
}).reset_index(drop=True)

In [85]:
result_df["model_comment"] = result_df["input_text"].apply(
    lambda x: explain_tfidf_pipeline(x, tfidf_model)
)

In [86]:
print(result_df.columns)

Index(['input_text', 'actual_label', 'predicted_label', 'confidence',
       'model_comment'],
      dtype='object')


In [87]:
# Actual split
result_df[["category", "subcategory"]] = result_df["actual_label"].str.split("_", n=1, expand=True)

# Predicted split
result_df[["predicted_category", "predicted_subcategory"]] = result_df["predicted_label"].str.split("_", n=1, expand=True)

In [88]:
result_df["confidence_percent"] = (result_df["confidence"] * 100).round(2).astype(str) + "%"

def confidence_level(c):
    if c >= 0.7:
        return "High"
    elif c >= 0.4:
        return "Medium"
    else:
        return "Low"

result_df["confidence_level"] = result_df["confidence"].apply(confidence_level)

In [89]:
result_df["correct"] = result_df["actual_label"] == result_df["predicted_label"]

In [90]:
def extract_fields(text):
    parts = text.split("|")
    narration = parts[0].replace("narration:", "").strip()
    mode = parts[1].replace("mode:", "").strip()
    txn_type = parts[2].replace("type:", "").strip()
    return pd.Series([narration, mode, txn_type])

result_df[["narration", "mode", "type"]] = result_df["input_text"].apply(extract_fields)

In [98]:
print(result_df.columns)

Index(['input_text', 'actual_label', 'predicted_label', 'confidence',
       'model_comment', 'category', 'subcategory', 'predicted_category',
       'predicted_subcategory', 'confidence_percent', 'confidence_level',
       'correct', 'narration', 'mode', 'type'],
      dtype='object')


In [99]:
final_df = result_df[[
    "narration",
    "mode",
    "type",
    "category",
    "subcategory",
    "predicted_category",
    "predicted_subcategory",
    "confidence",
    "confidence_percent",
    "confidence_level",
    "correct",
    "model_comment"
]]

final_df.head(20)

,narration,mode,type,category,subcategory,predicted_category,predicted_subcategory,confidence,confidence_percent,confidence_level,correct,model_comment
0,interest from fixed deposit,neft,credit,investment,fd_interest,investment,fd_interest,0.440036,44.0%,Medium,True,"Words [from, fixed, deposit] influenced predic..."
1,fd interest payout,neft,credit,investment,fd_interest,investment,fd_interest,0.462997,46.3%,Medium,True,"Words [payout, interest, fd] influenced predic..."
2,fixed deposit interest,neft,credit,investment,fd_interest,investment,fd_interest,0.498058,49.81%,Medium,True,"Words [fixed, deposit, interest] influenced pr..."
3,bank interest received,neft,credit,investment,fd_interest,investment,fd_interest,0.454949,45.49%,Medium,True,"Words [received, bank, interest] influenced pr..."
4,fd maturity interest,neft,credit,investment,fd_interest,investment,fd_interest,0.469820,46.98%,Medium,True,"Words [maturity, interest, fd] influenced pred..."
5,bank fd interest credit,neft,credit,investment,fd_interest,investment,fd_interest,0.515716,51.57%,Medium,True,"Words [credit, bank, interest] influenced pred..."
6,fd interest received,neft,credit,investment,fd_interest,investment,fd_interest,0.517287,51.73%,Medium,True,"Words [received, interest, fd] influenced pred..."
7,roi amount credited,neft,credit,investment,fd_interest,investment,fd_interest,0.223712,22.37%,Low,True,"Words [roi, amount, credited] influenced predi..."
8,term deposit interest,neft,credit,investment,fd_interest,investment,fd_interest,0.455252,45.53%,Medium,True,"Words [term, deposit, interest] influenced pre..."
9,fd interest deposit,neft,credit,investment,fd_interest,investment,fd_interest,0.566850,56.69%,Medium,True,"Words [deposit, interest, fd] influenced predi..."


In [92]:
from sklearn.metrics import classification_report

report = classification_report(y_test, y_pred_tfidf, output_dict=True)
report_df = pd.DataFrame(report).transpose()

In [93]:
def f2_score(p, r):
    if (p + r) == 0:
        return 0
    return (5 * p * r) / (4 * p + r)

report_df["f2_score"] = report_df.apply(
    lambda row: f2_score(row["precision"], row["recall"]),
    axis=1
)

report_df = report_df.round(3)

print("\nClassification Report:")
print(report_df)


Classification Report:
                        precision  recall  f1-score  support  f2_score
expense_bills               0.875   0.875     0.875     8.00     0.875
expense_food                1.000   0.875     0.933     8.00     0.897
expense_health              1.000   1.000     1.000     7.00     1.000
expense_insurance           1.000   1.000     1.000     8.00     1.000
expense_loan                1.000   1.000     1.000    12.00     1.000
expense_shopping            1.000   1.000     1.000    12.00     1.000
expense_transport           0.909   1.000     0.952    10.00     0.980
income_salary               1.000   1.000     1.000     8.00     1.000
investment_fd_booking       1.000   1.000     1.000     8.00     1.000
investment_fd_interest      1.000   1.000     1.000    10.00     1.000
investment_stocks           1.000   1.000     1.000     8.00     1.000
accuracy                    0.980   0.980     0.980     0.98     0.980
macro avg                   0.980   0.977     0.978  

In [94]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred_tfidf)

print("\nOverall Metrics:")
print("Accuracy:", round(accuracy, 3))
print("Macro F1:", report_df.loc["macro avg", "f1-score"])
print("Weighted F1:", report_df.loc["weighted avg", "f1-score"])


Overall Metrics:
Accuracy: 0.98
Macro F1: 0.978
Weighted F1: 0.98


In [95]:
# Remove summary rows
filtered_df = report_df.drop(["accuracy", "macro avg", "weighted avg"], errors="ignore")

best = filtered_df["f1-score"].idxmax()
worst = filtered_df["f1-score"].idxmin()

print("\nBest Performing Category:")
print(best, filtered_df.loc[best, "f1-score"])

print("\nWorst Performing Category:")
print(worst, filtered_df.loc[worst, "f1-score"])


Best Performing Category:
expense_health 1.0

Worst Performing Category:
expense_bills 0.875
